Ноутбук служит основой для изучения, однако многое нужно будет поискать и почитать самостоятельно. Никто вас не ограничивает в наведении красоты, но требуется выполнение того, что в задании.
***

Цель этой домашней работы все еще не научиться делать классные модели, а **подготавливать данные так, чтобы самая простая модель работала хорошо**.

В этом задании поговорим про работу с фичами, использовать можно только то, что указано в задании. НЕЛЬЗЯ МЕНЯТЬ ПАРАМЕТРЫ МОДЕЛЕЙ, нужно их использовать так, как они заданы.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor
from sklearn.metrics import accuracy_score

from typing import List, Dict

# Подготовка данных (1 pt)

## Теория

В этом задании мы уже не будем говорить о подготовке данных, как о проверке данных датасета на пропуски, выбросы и т.д. — вы уже это умеете. Тут мы поговорим о подготовке данных как о их **структуризации для обучения модели**, о **разбиении данных**.

**Три выборки — Train, Valid, Test**

Зачем нужно именно три выборки, а не две? Иногда действительно достаточно разбить данные просто на _train_ и _test_. Но этого может быть недостаточно для серьёзной работы. У каждой из трех выборок есть весьма логичная функция, которая важна для обучени:
* **train** выборка — это те данные, которая видит наша модель и на которой подбираются нужные коэффициенты и веса (функция — буквально **обучение**);
* **valid** выборка — это ваша «внутренняя» проверка, на которой вы в процессе обучения проверяете модель и настраиваете гиперпараметры (функция — **настройка** модели и **валидация**);
* **test** выборка же нужна только для итоговой финальной проверки, вы ее используете один раз в конце (функция — **объективная оценка качества**).

Очевидно (как мы и говорили в прошлой дз), что с test выборкой нельзя делать никаких обработок, связанных с обучением. Если вы хотя бы раз посмотрите на test-данные в процессе настройки модели, вы неосознанно начнёте подстраиваться под них. Это называется **data leakage** (утечка данных). Модель покажет отличные результаты на test, но провалится в реальном применении.

Логичным разбиением для теста можно считать взятие **последних данных**, если они **временные**, или использование **случайного подмножества** всех данных ([`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)). 

***

Еще одно важное для нас в этой работе понятие — это **data drift**.

**Что такое Data Drift?**

Data Drift — это **изменение статистических свойств** данных между выборками (например, между train и valid). Представьте, что вы обучили модель предсказывать стоимость страховки на данных 2020 года. А в 2024 году цены выросли, демография изменилась, появились новые факторы риска. **Распределение данных изменилось** — это и есть _data drift_.

Если очень хочется, то можно выделить несколько видов дрифта:
* **Covariate Drift** — меняется распределение признаков;
* **Label Drift** — меняется распределение целевой переменной;
* **Concept Drift** — меняется связь между признаками.

Третий тип сложно обсуждать без реального наблюдения таких данных, поэтому мы будем трогать только первые два.  
В этом задании мы будем стараться не допустить (отследить) _data drift_ на всех выборках. 

**Для обнаружения дрифта можно использовать определенные [метрики](https://habr.com/ru/companies/plarium/articles/515178/)**

**PSI (Population Stability Index)**

**PSI** — это метрика, которая показывает, насколько изменилось распределение признака между двумя выборками.  
Для расчета данные разбиваются на группы (бины), и для каждого бина сравниваются доли объектов в обучающей (_Expected_) и текущей (_Actual_) выборках по формуле:

$$PSI = \sum_{i=1}^{n} \left( (\% \text{Actual}_i - \% \text{Expected}_i) \cdot \ln\left( \frac{\% \text{Actual}_i}{\% \text{Expected}_i} \right) \right)$$

> Индекс PSI < 0,1 — без изменений.  
> Индекс PSI >= 0,1, но меньше 0,2 — требуются небольшие изменения.  
> PSI >= 0,2 — требуются значительные изменения.

**CSI (Characteristic Stability Index)**

**CSI** — это PSI, рассчитанный для категориальных признаков. Принцип тот же, но вместо квантильных бинов мы используем естественные категории. 

***

Помимо таких метрик можно использовать **статистические тесты** для проверки гипотез.

**KS-тест ([Kolmogorov-Smirnov](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ks_2samp.html))**

KS-тест отвечает на вопрос: «Можно ли считать, что две выборки взяты из одного распределения?»

> $H_0$ (нулевая гипотеза): Распределения одинаковы (drift нет)  
> $H_1$ (альтернативная гипотеза): Распределения различаются (drift есть)

_Правило принятия решения_  
p-value $<$ 0.05: Отклоняем, но есть статистически значимый drift.  
p-value $\geqslant$ 0.05: Не отклоняем, но drift не обнаружен.

**t-тест [Стьюдента](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html)**

Используется для сравнения средних значений между двумя выборками.
Лучше применять, когда признак распределён нормально (или выборка большая).

> $H_0$: Средние равны $(\mu_1 = \mu_2)$  
> $H_1$: Средние не равны $(\mu_1 \neq \mu_2)$

ПОДРОБНЕЕ НА МАТСТАТЕ
***

Еще одной классной фишкой для разбиения данных является **K-fold**. это стратегия обучения, которая очень полезна для малого количества данных. Так как это уже скорее отночится именно к стратегиям обучения, то мы не будем ее изучать, но ссылки на [документацию](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html),
[еще документацию](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
и [текст](https://neerc.ifmo.ru/wiki/index.php?title=%D0%9A%D1%80%D0%BE%D1%81%D1%81-%D0%B2%D0%B0%D0%BB%D0%B8%D0%B4%D0%B0%D1%86%D0%B8%D1%8F) 
я оставлю.

## Задание (1 pt)

В этом и следующих нескольких заданиях мы поработаем с датасетом `Medical Insurance Cost` и задачей регресии (предсказание `charges`). Он довольно маленький, но для обучения сгодится (вот для него бы k-fold был кстати).

In [2]:
df = pd.read_csv('insurance.csv')
print(df.shape)
df.head()

(1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [ ]:
# быстрый обзор и базовая обработка, можете не комментировать и не оставлять ресерческий код
raise NotImplementedError

Для начала разбейте данные на три выборки **train**, **valid** и **test** в соотношении (65/15/20). В `train_test_split` есть замечательный параметр `stratify`, который помогает делить данные равномерно, используйте его (но он работает **только с категориальными признаками**).  
НЕ МЕНЯЙТЕ СИДЫ

In [ ]:
# ЗАДАНИЕ: разбить датасет на три выборки (можно сделать функцию)

# для stratify стратифицируем по квантилям
charges_bins = pd.qcut(df['charges'], q=..., labels=False, duplicates='drop')

train_valid, test = train_test_split(
    df, 
    test_size=..., 
    random_state=42, 
    stratify=...
)

# пересчитываем квантили для оставшихся данных
charges_bins_tv = pd.qcut(train_valid['charges'], q=..., labels=False, duplicates='drop')

train, valid = train_test_split(
    train_valid, 
    test_size=...,
    random_state=42, 
    stratify=...
)

# сохраните на всякий случай, чтобы потом не пересчитывать
# train.to_csv('train.csv', index=False)
# valid.to_csv('valid.csv', index=False)
# test.to_csv('test.csv', index=False)

Теперь нужно посмотреть, нормальное ли у нас разбиение получилось

In [ ]:
# ЗАДАНИЕ: реализуйте PSI и CSI

def calculate_psi(expected, actual, buckets=10):
    # создаём бины на основе expected распределения
    breakpoints = np.percentile(expected, np.linspace(..., ..., buckets + 1))
    breakpoints = np.unique(breakpoints)
    
    # распределяем данные по бинам
    expected_counts = np.histogram(..., bins=...)[0]
    actual_counts = np.histogram(..., bins=...)[0]
    
    # конвертируем в проценты
    expected_pct = ... / len(...)
    actual_pct = ... / len(...)
    
    # добавляем epsilon для избежания деления на 0
    epsilon = 0.0001
    expected_pct = np.clip(expected_pct, epsilon, 1)
    actual_pct = np.clip(actual_pct, epsilon, 1)
    
    # рассчитываем PSI по формуле
    psi_contributions = (... - ...) * np.log(... / ...)
    psi = np.sum(psi_contributions)

    return psi


def calculate_csi(expected, actual):
    # получаем уникальные категории
    all_categories = np.union1d(np.unique(...), np.unique(...))
    
    # считаем проценты по категориям
    expected_pct = np.array([np.sum(... == cat) / len(...) for cat in ...])
    actual_pct = np.array([np.sum(... == cat) / len(...) for cat in ...])
    
    # добавляем epsilon для избежания деления на 0
    epsilon = 0.0001
    expected_pct = np.clip(expected_pct, epsilon, 1)
    actual_pct = np.clip(actual_pct, epsilon, 1)
    
    # рассчитываем CSI по формуле PSI
    csi_contributions = (... - ...) * np.log(... / ...)
    csi = np.sum(csi_contributions)
    
    return csi

In [ ]:
# ПРОВЕРКА, НЕ МЕНЯЙТЕ КОД

numerical_features = ['age', 'bmi', 'children', 'charges']
categorical_features = ['sex', 'smoker', 'region']

for feature in numerical_features:
    psi_tv = calculate_psi(train[feature].values, valid[feature].values)
    psi_tt = calculate_psi(train[feature].values, test[feature].values)

    assert psi_tv < 0.1
    assert psi_tt < 0.1
    
    print(f"{feature} PSI_tv = {psi_tv:.4f}, PSI_tt = {psi_tt:.4f}")

print()

for feature in categorical_features:
    csi_tv = calculate_csi(train[feature].values, valid[feature].values)
    csi_tt = calculate_csi(train[feature].values, test[feature].values)

    assert csi_tv < 0.1
    assert csi_tt < 0.1
    
    print(f"{feature} CSI_tv = {csi_tv:.4f}, CSI_tt = {csi_tt:.4f}")

Теперь больше для обучения, чем для уверенности проведем еще статистические тесты

In [ ]:
# ЗАДАНИЕ: проведите статистические тесты для числовых признаков и сделайте вывод

numerical_features = ['age', 'bmi', 'children', 'charges']

# KS-test
for feature in numerical_features:
    ks_statistic, ks_pvalue = ...
    
    ...

# T-test
for feature in numerical_features:
    t_statistic, t_pvalue = ...
    
    ...


**ВАШ ВЫВОД**   
пары слов достаточно

# Статистическое тестирование признаков и Feature Engineering (2 pt)

## Теория

После разбиения данных мы можем перейти к **работе с признаками** (чтобы утечки данных не было).

Очевидно, что для предсказания **не все признаки одинаково важны**, все они по-разному связаны с целевой переменной. Поэтому было бы хорошо **определять важные признаки** и использовать для обучения именно их. 

В этой части мы будем работать именно со **статистическими методами** определения важности и влияния (некоторые вы уже даже знаете).  
Для разных типов признаков, конечно, можно использовать разные методы, давайте их рассмотрим.  
_Я поняла, что в документацию тыкают редко, поэтому тут ищите ее сами :)_

**Для числовых признаков**

| Метод | Код | Что показывает |
| :---- | :-- | :------------- |
| **Корреляция Пирсона** | `stats.pearsonr()` | Силу линейной связи |
| **Корреляция Спирмена** | `stats.spearmanr()` | Силу монотонной связи |
| **t-тест** | `stats.ttest_ind()` | Различаются ли средние |
| **ANOVA** | `stats.f_oneway()` | Различаются ли средние групп |

**Для категориальных признаков**

| Метод | Код | Что показывает |
| :---- | :-- | :------------- |
| **Хи-квадрат** | `stats.chi2_contingency()` | Есть ли зависимость между признаками |
| **[Cramér's V](https://en.wikipedia.org/wiki/Cram%C3%A9r%27s_V)** | через $\chi^2$ | Сила связи категориальных признаков |
| **t-тест/ANOVA** | `stats.ttest_ind()`, `stats.f_oneway()` | Влияет ли категория на target |

***

Помимо определения важности уже существующих признаков нужно создавать хорошие новые! Логично, что просто создать хорошие новые нереально. Поэтому надо **создавать логически (или математически) подходящие** и оценивать их. 

Создание и вообще работа с признаками называется крутыми словами **Feature Engineering**. Про какое именно создание идет речь:
* **доменные признаки** — создание нового признака из бизнес-логики или доманного знания (какая-нибудь физическая формула например);
* **математические признаки** — квадраты, корни, логорифмы, произведения существующих признаков с целью уловить нелинейные зависимости;
* **агрегации** — группировки в данных и подсчет статистик через это (gb и все такое);
* **бинарные признаки** — отметки важных порогов из бизнес-логики или домееных знаний (например минимальная сумма на выдачу ипотеки или что-то такое).

Очевидно, что простые модели _не всегда могут уловить сложные зависимости_. Поэтому, создавая _нелинейные новые признаки_, мы можем дать модели очень полезную информацию.

## Задание (2 pt)

Для начала подготовим вспомогательную функцию, которая потом нам поможет оценивать важность и влияние признаков на целевую переменную.

In [ ]:
# ЗАДАНИЕ: напишите функцию для оценки признаков
#          функция должна возвращять словарь с подсчитанными статистиками

# feature_types : dict {feature_name: 'numerical' или 'categorical'}
def assess_feature_significance(df: pd.DataFrame, features: List[str], feature_types: Dict[str,str], target_col: str):

    results = {}
    
    for feature in features:
        f_type = feature_types.get(feature, 'numerical')
        
        if f_type == 'numerical':
            
            p_corr = ...
            s_corr = ...

            results[feature] = {'p_corr' : p_corr, 's_corr' : s_corr}
            
        elif f_type == 'categorical':
            
            # проверяем количество уникальных значений
            n_categories = df[feature].nunique()
            
            if n_categories == 2:
                # t-тест
                groups = df.groupby(feature)[target_col]
                group_values = [group.values for name, group in groups]
                stat, p_value = ...
                
                # размер эффекта: Cohen's d
                pooled_std = np.sqrt((group_values[0].var() + group_values[1].var()) / 2)
                cohens_d = abs(group_values[0].mean() - group_values[1].mean()) / pooled_std
                effect_size = cohens_d
                
            else:
                # ANOVA
                groups = df.groupby(feature)[target_col]
                group_values = [group.values for name, group in groups]
                stat, p_value = ...

                # размер эффекта: Eta-squared
                ss_between = sum(len(g) * (g.mean() - df[target_col].mean())**2 for g in group_values)
                ss_total = sum((df[target_col] - df[target_col].mean())**2)
                eta_squared = ss_between / ss_total
                effect_size = eta_squared
            

            results[feature] = {'p_value' : p_value, 'effect_size' : effect_size}
    
    return results

In [ ]:
feature_types = {
    'age': 'numerical',
    'bmi': 'numerical',
    'children': 'numerical',
    'sex': 'categorical',
    'smoker': 'categorical',
    'region': 'categorical'
}

effect_size = assess_feature_significance(
    df=train,
    features=list(feature_types.keys()),
    feature_types=feature_types,
    target_col='charges',
)

for feature, stats_dict in significance_results.items():
    feature_stats = significance_results[feature]
    if 'p_corr' in feature_stats:
        print(f"{feature}: p_corr {feature_stats['p_corr']:.2f}, s_corr {feature_stats['s_corr']:.2f}")
    else:
        print(f"{feature}: p_value {feature_stats['p_value']:.2f}, effect_size {feature_stats['effect_size']:.2f}")

**ВЫВОД** пару слов о признаках

In [ ]:
# ЗАДАНИЕ: создайте 2-3 новых признака (можно и пороговых), основынных на бизнес или доменной логике и посмотрите на их влияние
#          в текстовом поле после кода напишите логическое объяснение этих признаков

df['new_...'] = ... # дайте признаку понятное и интерпретируемое имя

raise NotImplementedError

**ОБЪЯСНЕНИЕ ПРИЗНАКОВ И ВЫВОД О ЗНАЧИМОСТИ**

In [ ]:
# ЗАДАНИЕ: создайте 7-10 новых математических признака и посмотрите на их влияние
#          используйте нелинейные комбинации, степени, логорифмирование и так далее

df['new_...'] = ... # дайте признаку понятное и интерпретируемое имя

raise NotImplementedError

**ВЫВОД О ЗНАЧИМОСТИ И ПОПЫТКА ИНТЕРПРЕТАЦИИ ЗНАЧИМЫХ**

# Важность признаков через модели (2 pt)

## Теория

Теперь мы умеем как-то оценивать важность признаков и создали новые признаки. Но, не все так просто. Нетрудно догадаться, что **важность признаков для модели очень зависит от самой модели**.

Но с другой стороны мы как раз можем **использовать модели и через них определять важность признаков**!

То есть логика всего действа такая: 
* определить **статистически** значимые признаки,
* обучить модель и с помощью нее получить **«практическую значимость»** признаков,
* **поменять набор признаков**, опираясь на знание важности, так, чтобы модель лучше работала.
_А можно вообще использовать базовые модели для определения важности, а само предсказание делать на более сложных._

Для разных моделей методы определения важности признаков свои, давайте быстро по ним пройдемся.

**1. Веса линейной (логистической) легрессии**

Можно напрямую посмотреть на то, **какой вес вносит каждый признак в предсказание** (будет работать, если была нормализация) `inear_model.coef_`. Очевидно, что нелинейные связи в этом случае не отслеживаются.

**2. L1 Регуляризация (Lasso)**

Это линейная регрессия сс регуляризацией (подробнее на мо или в интернетах) `asso_model.coef_`. Она напротив **«обнуляет» неважные признаки**. В том числе она может обнулить коррелированные признаки или те, которые сильно влияют друг на друга.

**3. Feature Importance (Tree-Based)**

Смторит на то, **как часто признак участвует в разбиениях** `rf_model.feature_importances_`. Не требует никакого масштабирования, но может сильно смещаться в сторону признаков с большим количеством разных значений. Если мы говорим о `CatBoost`, то можно без обработки смотреть на важность категориальных признаков.

_Если ваша цель — жестко нафичеинжинирить, то лучше, конечно, использовать все способы, сравнивать и комбинировать результаты._

## Задание (2 pt)

Давайте обучим разные модели и посмотрим на важность признаков в каждой из них.  
ПАРАМЕТРЫ МОДЕЛЕЙ МЕНЯТЬ НЕЛЬЗЯ, ВАША СВОБОДА — ЭТО ПРИЗНАКИ  

ДА, МЫ НЕ БУДЕМ ТУТ ИСПОЛЬЗОВАТЬ ВАЛИДАЦИЮ, ПОТОМУ ЧТО ВОТ ТАК, ОНА БЫЛА ЧИСТО УЧЕБНОЙ ЧАСТЬЮ

In [ ]:
# НЕ МЕНЯТЬ
RANDOM_STATE = 42
LASSO_ALPHA = 0.1
RF_N_ESTIMATORS = 100
RF_MAX_DEPTH = 5
CATBOOST_ITERATIONS = 100
CATBOOST_DEPTH = 4
CATBOOST_LEARNING_RATE = 0.1

In [ ]:
# ЗАДАНИЕ: обработайте категориальные данные для моделей (учитесь на test)
#          не забудьте про масштабирование для линейной модели

raise NotImplementedError

In [ ]:
# ЗАДАНИЕ: обучите 4 модели 

linear_model = LinearRegression()
linear_model.fit(..., ...)
linear_score = linear_model.score(..., ...)
print(f"LinearRegression: {linear_score:.4f}")


lasso_model = Lasso(alpha=LASSO_ALPHA, random_state=RANDOM_STATE, max_iter=10000)
lasso_model.fit(..., ...)
lasso_score = lasso_model.score(..., ...)
n_zero_coef = np.sum(lasso_model.coef_ == 0)
print(f"Lasso: {lasso_score:.4f}")
print(f"Обнулено коэффициентов: {n_zero_coef} из {len(lasso_model.coef_)}")


rf_model = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(..., ...)
rf_score = rf_model.score(..., ...)
print(f"RandomForestRegressor: {rf_score:.4f}")


catboost_model = CatBoostRegressor(
    iterations=CATBOOST_ITERATIONS,
    depth=CATBOOST_DEPTH,
    learning_rate=CATBOOST_LEARNING_RATE,
    random_state=RANDOM_STATE,
    verbose=0
)
catboost_model.fit(..., ...)
catboost_score = catboost_model.score(..., ...)
print(f"CatBoostRegressor: {catboost_score:.4f}")

In [ ]:
# ЗАДАНИЕ: получите важность признаков для каждой модели
#          визуализируйте, сравните и напишите вывод

linear_importance = np.abs(...)
linear_importance_dict = dict(zip(all_features, linear_importance))


lasso_importance = np.abs(...)
lasso_importance_dict = dict(zip(all_features, lasso_importance))


rf_importance = ...
rf_importance_dict = dict(zip(all_features, rf_importance))


catboost_importance = ...
catboost_importance_dict = dict(zip(all_features, catboost_importance))

importance_df = pd.DataFrame({
    'Feature': all_features,
    'Linear_Regression': linear_importance,
    'Lasso': lasso_importance,
    'Random_Forest': rf_importance,
    importance_df['CatBoost'] = catboost_importance
})
    
# нормализуем важность для сравнения
for col in ['Linear_Regression', 'Lasso', 'Random_Forest', 'CatBoost']:
    if col in importance_df.columns:
        max_val = importance_df[col].max()
        if max_val > 0:
            importance_df[col + '_norm'] = importance_df[col] / max_val

# визуализация
raise NotImplementedError

**ВЫВОД**

***

Теперь вы можете поэксперементировать с наборами признаков, чтобы получить лучшую метрику на тесте

In [ ]:
# ЗАДАНИЕ: выберите одну модель, которой будете получать итоговое предсказание (R^2 метрика)
#          поэксперементируйте с наборами признаков и предъявите:
#              1. набор признаков и результаты обучения и теста с самым ПЛОХИМ скором
#              2. набор признаков и результаты обучения и теста с самым ХОРОШИМ скором

raise NotImplementedError

**ПАРА СЛОВ ОБ ИТОГАХ**

# Практическое использование навыков (5 pt)

_В этот раз мы снова поработаем с вами с красивыми, но невкусными напитками._  

Возьмем датасет `Wine Quality` и задачу предсказания качества вина на основе физико-химических свойств. Целевой столбец — `quality`.

In [3]:
wine = pd.read_csv('WineQT.csv')
print(wine.shape)
wine.head()

(1143, 13)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


Нетрудно заметить, что это задача **многоклассовой классификации**.

***
**ЗАДАНИЕ**

Вам необходимо поработать с признаками и обучить случайный лес с фиксированными параметрами, которые нельзя менять.  
Необходимо достичь метрики **Accuracy** > 0.7 на тесте и **явно это показать** (это примерно на 3% больше, чем базово из коробки).

_**РАЗРЕШЕНО**_  

Эксперементировать с признаками и применять знания из превой части домашки

_**ЗАПРЕЩЕНО**_

Использовать другие модели и менять заданные константы, использовать другие параметры модели или какие-то неочевидные ивристики.

**ОЦЕНИВАЕТСЯ**
* Подготовка и обработка данных
* Анализ важности признаков
* Создание новых признаков и их отбор (в том числе их объяснение и интерпретация)
* Обучение и получение итоговой метрики
  
***

In [ ]:
RANDOM_STATE = 42 # и для разбиения тоже
RF_N_ESTIMATORS = 150
RF_MAX_DEPTH = 10

model = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

TEST_SIZE = 0.2 # для деления на train/test

In [ ]:
...

In [ ]:
# НЕ МЕНЯТЬ
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y
)

In [ ]:
...

In [ ]:
y_pred = model.predict(X_test)
score = accuracy_score(y_test, y_pred)
assert score > 0.7
print(f"Accuracy (Multi-class): {score:.4f}")

**ОТЧЕТ О ПРОДЕЛАННОЙ РАБОТЕ** достаточно описания ваших попыток и того, к чему в итоге пришли